# Quantum Critical Point with Fermi Statistics and SCF

This notebook computes the **quantum** critical-point quantities `T_c` and `n_c`.

It leaves the existing ground-state-only package unchanged and reuses its already-working
ground-state fit for `a`, `b`, and `K_0`.


## Mathematical Setup

For each interaction model, the ground-state package provides the fitted parameters
`a` and `b`. At finite temperature we then use the **quantum** ideal Fermi gas.

### 1. Ideal-gas finite-temperature quantities

The relativistic single-particle energy is

$$
E(k) = \sqrt{(\hbar c\,k)^2 + m^2}.
$$

The Fermi-Dirac occupation factor is

$$
f(E,\mu^*,T) = \frac{1}{e^{(E-\mu^*)/T}+1}.
$$

Then the ideal density and pressure are computed numerically as

$$
n^{\mathrm{id}}(T,\mu^*) = \frac{d}{2\pi^2} \int_0^{k_{\max}} k^2 f(E,\mu^*,T)\,dk,
$$

$$
p^{\mathrm{id}}(T,\mu^*) = \frac{d}{6\pi^2} \int_0^{k_{\max}} \frac{(\hbar c)^2 k^4}{E(k)} f(E,\mu^*,T)\,dk.
$$

### 2. Excluded-volume map

For a physical density `n`, the corresponding ideal density is

$$
n^{\mathrm{id}} = \frac{n}{1-bn}.
$$

Therefore, for each pair `(T,n)`, we solve the scalar self-consistency condition

$$
n^{\mathrm{id}}(T,\mu^*) = \frac{n}{1-bn}
$$

for the effective chemical potential `\mu^*`.

### 3. SCF update for `\mu^*`

We define the residual

$$
R_\mu(\mu^*) = n^{\mathrm{id}}(T,\mu^*) - \frac{n}{1-bn},
$$

and iterate with a damped SCF correction

$$
\mu^*_{m+1} = \mu^*_m - \lambda_\mu \frac{R_\mu(\mu^*_m)}{\chi_\mu(\mu^*_m)},
$$

where the response is estimated numerically by

$$
\chi_\mu \approx \frac{n^{\mathrm{id}}(T,\mu^*+\delta\mu)-n^{\mathrm{id}}(T,\mu^*-\delta\mu)}{2\delta\mu}.
$$

### 4. Quantum pressure as a function of `(T,n)`

Once `\mu^*` is found, the quantum pressure is evaluated as

$$
P(T,n) = p^{\mathrm{id}}(T,\mu^*) + a n^2 U'(n).
$$

### 5. Critical-point equations

The quantum critical point is defined by

$$
F_1(T,n) = \left(\frac{\partial P}{\partial n}\right)_T = 0,
$$

$$
F_2(T,n) = \left(\frac{\partial^2 P}{\partial n^2}\right)_T = 0.
$$

Both derivatives are evaluated numerically using centered finite differences in `n`.

### 6. Outer SCF solve for `(T_c,n_c)`

Starting from a coarse seed, we perform a damped SCF iteration:

$$
T_{k+1} = T_k - \lambda_T \frac{F_1(T_k,n_k)}{\partial F_1/\partial T},
$$

$$
n_{k+1} = n_k - \lambda_n \frac{F_2(T_k,n_k)}{\partial F_2/\partial n},
$$

where the slopes `\partial F_1/\partial T` and `\partial F_2/\partial n` are also computed numerically.

This gives a fully numerical quantum solution without using external nonlinear solvers.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
project_dir = None

for candidate in [cwd, cwd.parent, cwd / "quantum_tc_nc_scf"]:
    if (candidate / "src" / "quantum_tc_nc_scf").exists():
        project_dir = candidate
        break

if project_dir is None:
    raise RuntimeError("Could not locate quantum_tc_nc_scf/src.")

ground_state_dir = project_dir.parent / "ground_state_ab_k0" / "src"
quantum_src_dir = project_dir / "src"
results_dir = project_dir / "results"
results_dir.mkdir(exist_ok=True)

for path in [str(ground_state_dir), str(quantum_src_dir)]:
    if path not in sys.path:
        sys.path.insert(0, path)

from quantum_tc_nc_scf import DEFAULT_ALPHA_LIST, DEFAULT_C_LIST, QuantumCriticalSettings, compute_model_family_quantum, compute_quantum_critical_point
from quantum_tc_nc_scf.reporting import critical_results_to_records


In [ ]:
# Use QuantumCriticalSettings() for a denser production run.
settings = QuantumCriticalSettings.quick()

alpha_list = DEFAULT_ALPHA_LIST
c_list = DEFAULT_C_LIST


In [ ]:
base_results = [
    compute_quantum_critical_point("vdw", settings=settings),
    compute_quantum_critical_point("rks", settings=settings),
    compute_quantum_critical_point("pr", settings=settings),
]
base_results = [result for result in base_results if result is not None]

clausius_results = compute_model_family_quantum("clausius", c_list, settings=settings)
dieterici_results = compute_model_family_quantum("dieterici", alpha_list, settings=settings)


In [ ]:
base_df = pd.DataFrame(critical_results_to_records(base_results))
clausius_df = pd.DataFrame(critical_results_to_records(clausius_results, parameter_name="c"))
dieterici_df = pd.DataFrame(critical_results_to_records(dieterici_results, parameter_name="alpha"))

combined_df = pd.concat([base_df, clausius_df, dieterici_df], ignore_index=True)

base_df.to_csv(results_dir / "base_models_quantum_tc_nc.csv", index=False)
clausius_df.to_csv(results_dir / "clausius_quantum_tc_nc.csv", index=False)
dieterici_df.to_csv(results_dir / "dieterici_quantum_tc_nc.csv", index=False)
combined_df.to_csv(results_dir / "quantum_tc_nc_scf_results.csv", index=False)

display(base_df[["model", "a", "b", "K0", "Tc", "nc", "Pc", "score"]])
display(clausius_df[["parameter_value", "a", "b", "K0", "Tc", "nc"]])
display(dieterici_df[["parameter_value", "a", "b", "K0", "Tc", "nc"]])


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

if not clausius_df.empty:
    ax.plot(clausius_df["K0"], clausius_df["Tc"], color="magenta", marker="o", label="Clausius")
if not dieterici_df.empty:
    ax.plot(dieterici_df["K0"], dieterici_df["Tc"], color="green", marker="o", label="Dieterici")

if not base_df.empty:
    color_map = {"vdw": "black", "rks": "red", "pr": "blue"}
    for _, row in base_df.iterrows():
        ax.scatter(row["K0"], row["Tc"], color=color_map.get(row["model"], "black"), s=80)
        ax.text(row["K0"] + 8, row["Tc"] + 0.05, row["model"].upper())

ax.set_xlabel(r"$K_0$ [MeV]")
ax.set_ylabel(r"$T_c$ [MeV]")
ax.set_title("Quantum Critical Temperature vs Incompressibility")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(results_dir / "quantum_tc_vs_k0.png", dpi=300)
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

if not clausius_df.empty:
    axes[0].plot(clausius_df["K0"], clausius_df["nc"], color="magenta", marker="o", label="Clausius")
    axes[1].plot(clausius_df["Tc"], clausius_df["nc"], color="magenta", marker="o", label="Clausius")

if not dieterici_df.empty:
    axes[0].plot(dieterici_df["K0"], dieterici_df["nc"], color="green", marker="o", label="Dieterici")
    axes[1].plot(dieterici_df["Tc"], dieterici_df["nc"], color="green", marker="o", label="Dieterici")

if not base_df.empty:
    color_map = {"vdw": "black", "rks": "red", "pr": "blue"}
    for _, row in base_df.iterrows():
        color = color_map.get(row["model"], "black")
        axes[0].scatter(row["K0"], row["nc"], color=color, s=80)
        axes[1].scatter(row["Tc"], row["nc"], color=color, s=80)

axes[0].set_xlabel(r"$K_0$ [MeV]")
axes[0].set_ylabel(r"$n_c$ [fm$^{-3}$]")
axes[0].set_title(r"$n_c$ vs $K_0$")

axes[1].set_xlabel(r"$T_c$ [MeV]")
axes[1].set_ylabel(r"$n_c$ [fm$^{-3}$]")
axes[1].set_title(r"$n_c$ vs $T_c$")

for axis in axes:
    axis.grid(alpha=0.25)
    axis.legend()

fig.tight_layout()
fig.savefig(results_dir / "quantum_nc_summary.png", dpi=300)
plt.show()
